# 04 - Encoding of categorical variables

**The idea in one sentence:** a `scikit-learn` model only knows how to do arithmetic, so **everything that is text has to be turned into numbers** — but without making up information that does not exist.

This matches **step 2 of `preprocess_data()`** in [`src/preprocessing.py`](../src/preprocessing.py).

---

### The problem, in 2 lines

```python
LogisticRegression().fit(X_with_text, y)
# ValueError: could not convert string to float: 'Cash loans'
```

The model does not know what `'Cash loans'` is. It needs a number.

### The path of this notebook

| Step | Question | Answer |
|---|---|---|
| **1** | Which columns are text and how many distinct values do they have? | We count the **cardinality** |
| **2** | What if it only has 2 values? | We give it **0 and 1** (`OrdinalEncoder`) |
| **3** | And if it has more than 2? | **One column per category** (`OneHotEncoder`) |
| **4** | How does it all fit together? | A 10-line function |

The whole notebook boils down to **a single decision**: *does this column have 2 values or more than 2?*. Everything else follows from that question.

In [ ]:
import sys
from pathlib import Path

# Lets us import `helpers.py` no matter where the notebook is opened from.
for _p in [Path.cwd(), Path.cwd() / "material_extra", Path.cwd().parent]:
    if (_p / "helpers.py").exists():
        sys.path.insert(0, str(_p))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import helpers

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

GREEN, BLUE, RED = "#4c9f70", "#4c72b0", "#d1495b"
print("Original dataset available:", helpers.dataset_disponible())

In [ ]:
from sklearn.model_selection import train_test_split

df = helpers.cargar_muestra(n=20_000)

X = df.drop(columns=["TARGET"])
y = df["TARGET"]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"train={X_train.shape}   val={X_val.shape}")

---

# Step 1 — How much variety does each text column have?

**Cardinality** = how many distinct values a column has.

- `NAME_CONTRACT_TYPE` → `Cash loans`, `Revolving loans` → cardinality **2**
- `NAME_FAMILY_STATUS` → `Married`, `Single`, `Widow`, ... → cardinality **5**

It is the only piece of information we need in order to decide which encoder to use.

In [ ]:
categorical = X_train.select_dtypes(include="object").columns
cardinality = X_train[categorical].nunique().sort_values()

cardinality.to_frame("distinct values")

### The same table, but looking at it

The green bars are the columns with **2 values**; the blue ones, those with **more than 2**. The dashed line is the border that decides the encoder.

In [ ]:
binary = list(cardinality[cardinality == 2].index)
multiclass = list(cardinality[cardinality > 2].index)

colors = [GREEN if n == 2 else BLUE for n in cardinality]

fig, ax = plt.subplots(figsize=(9, 0.45 * len(cardinality) + 1.5))
ax.barh(cardinality.index, cardinality.values, color=colors)
ax.axvline(2.5, color="grey", linestyle="--", linewidth=1)

for name, n in cardinality.items():
    ax.text(n + 0.1, name, str(n), va="center", fontsize=9)

ax.set_xlabel("number of distinct categories")
ax.set_title("How many categories does each text column have?")
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color=GREEN, label="2 categories → OrdinalEncoder (0/1)"),
        plt.Rectangle((0, 0), 1, 1, color=BLUE, label="more than 2 → OneHotEncoder"),
    ],
    loc="lower right",
)
plt.tight_layout()
plt.show()

print(f"Binary     ({len(binary)}): {binary}")
print(f"Multiclass ({len(multiclass)}): {multiclass}")

---

# Step 2 — Columns with 2 categories: simply 0 and 1

With only two possible values there is no way to get it wrong: it does not matter which one is the 0 and which one is the 1, because **between two values no "fake ordering" is possible**. It is just a switch: off / on.

`OrdinalEncoder` does exactly that: it sorts the categories alphabetically and assigns them `0, 1, 2, ...`

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ord_enc = OrdinalEncoder()
ord_enc.fit(X_train[binary])                     # <-- fit ONLY on train

X_train_bin = X_train.copy()
X_val_bin = X_val.copy()
X_train_bin[binary] = ord_enc.transform(X_train[binary])
X_val_bin[binary] = ord_enc.transform(X_val[binary])   # <-- transform only

print("Dictionary the encoder learned:\n")
for col, cats in zip(binary, ord_enc.categories_):
    print(f"  {col:22s} {dict(zip(cats, range(len(cats))))}")

### Before and after, side by side

In [ ]:
before = X_train[binary].head(6)
after = X_train_bin[binary].head(6).astype(int)

comparison = pd.concat(
    [before, after],
    axis=1,
    keys=["BEFORE (text)", "AFTER (numbers)"],
)
comparison

`FLAG_OWN_CAR` used to be `Y`/`N` and now it is `1`/`0`: same information, different language. We neither lost nor invented anything.

### ❓ The question you are going to be asked

> *"But `OrdinalEncoder` is also imposing an order. Why is it fine here and not in step 3? And why don't we just use one-hot for everything?"*

It is the right question. The answer: **yes, the ordinal always places the categories on a line — but with 2 categories that line says nothing.**

The problem with the ordinal encoding for 3 or more categories is not the order itself, it is the **relative distances**. With 3 points on a line (`0, 1, 2`) one of them inevitably ends up **in the middle**, and the model assumes that one is *closer* to one than to the other. That is invented information.

**With 2 points there is no possible middle.** Any two points are, by definition, at the same distance from each other. So `{N:0, Y:1}` and `{N:1, Y:0}` have to produce exactly the same model. Let's check it with the three possible encodings:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

col = X_train["FLAG_OWN_CAR"]

# The SAME binary column, encoded in three different ways.
options = {
    "ordinal  N=0, Y=1": pd.DataFrame({"x": (col == "Y").astype(int)}),
    "ordinal  N=1, Y=0": pd.DataFrame({"x": (col == "N").astype(int)}),  # flipped
    "one-hot  (2 columns)": pd.get_dummies(col).astype(int),
}

for name, Xb in options.items():
    model = LogisticRegression().fit(Xb, y_train)
    proba = model.predict_proba(Xb)[:, 1]
    print(f"{name:22s}  AUC={roc_auc_score(y_train, proba):.6f}   "
          f"coefficients={np.round(model.coef_[0], 4)}")

**All three are the same model**: same AUC, same predictions. Flipping the encoding only changes the **sign** of the coefficient, and one-hot splits that same effect across two columns (`+β/2` and `−β/2`).

### So, could I use one-hot for the binary ones too?

**Yes, and nothing breaks** — the table above shows it. But it is **redundant**:

- The two columns always add up to 1 (`is_Y + is_N = 1`). The second one adds nothing: you already know its value by looking at the first. That is called **perfect collinearity** (or the *dummy variable trap*).
- That is why `OneHotEncoder(drop="first")` exists: a variable with **k** categories only needs **k − 1** columns. With k = 2, that is **a single column**... which is exactly what `OrdinalEncoder` returns.
- In other words: **the ordinal encoding of a binary column *is* one-hot done right**, without the leftover column.

With 4 binary columns the difference is 4 columns: nothing dramatic. The reason is conceptual, not about performance — we choose the **minimal** representation that neither loses nor invents information.

In short: **with two categories the number is not an order, it is a switch.**

### The nuance that does matter: it depends on the model

When the ordinal encoding **is** a mistake (3+ categories with no real order), the damage is not the same for everyone:

| Model | How bad it is |
|---|---|
| **Logistic regression / linear models** | Bad: the coefficient **multiplies** the number, so it literally assumes that category `3` "weighs three times" what category `1` does. |
| **Trees / Random Forest** | Less bad, but limiting: the tree can only split at thresholds along that line, so it can only group categories that are **contiguous in an arbitrary alphabetical order**. What one-hot solves with a single split sometimes costs it several. |

---

# Step 3 — Columns with more than 2 categories

### First: why do we NOT give them 0, 1, 2, 3?

This is the key question of the notebook. If we assign numbers to `NAME_INCOME_TYPE`, the model —which only knows how to do arithmetic— assumes those numbers **mean something**: that they are ordered and that the distances between them are real.

Let's draw it.

In [ ]:
# We take the 4 most frequent categories so the drawing reads well.
cats = sorted(X_train["NAME_INCOME_TYPE"].value_counts().head(4).index)
codes = np.arange(len(cats))

fig, ax = plt.subplots(figsize=(10, 2.6))
ax.hlines(0, -0.5, len(cats) - 0.5, color="lightgrey", linewidth=3, zorder=1)
ax.scatter(codes, np.zeros(len(cats)), s=300, color=RED, zorder=3)

for i, c in enumerate(cats):
    ax.text(i, 0.22, c, ha="center", fontsize=10)
    ax.text(i, -0.28, str(i), ha="center", fontsize=13, weight="bold", color=RED)

ax.annotate(
    "", xy=(0, -0.62), xytext=(3, -0.62),
    arrowprops=dict(arrowstyle="<->", color="grey"),
)
ax.text(1.5, -0.55, "the model believes there are 3 units of distance between these two",
        ha="center", fontsize=9, color="grey")

ax.set_ylim(-0.9, 0.55)
ax.set_xlim(-0.7, len(cats) - 0.3)
ax.axis("off")
ax.set_title("What the model 'understands' if we number the categories", fontsize=12)
plt.tight_layout()
plt.show()

### The three lies in the drawing

By placing them on a number line, the model assumes without anyone telling it:

1. **Order:** that the last category is *"greater"* than the first.
2. **Distance:** that the gap between 0 and 1 is *the same* as the one between 1 and 2.
3. **Averages:** that the midpoint between two categories (a `1.5`) means something.

None of the three is true: they are **labels without order** (*nominal* variables). One income type is not "more" than another.

> **An analogy:** if you number the colors `red=0, green=1, blue=2`, the model concludes that *"green is the average between red and blue"*. Nobody would say that out loud, but it is exactly what we are teaching it.

---

### The solution: `OneHotEncoder` — swapping one open question for several yes/no ones

`NAME_INCOME_TYPE` is **one** text column that can take 6 different values. One-hot replaces it with **6 new columns, one per possible value**.

Put another way: instead of asking *"what income type do they have?"* (answer: text), we now ask 6 yes/no questions:

- *is it `Pensioner`?* → 0 or 1
- *is it `Working`?* → 0 or 1
- *is it `State servant`?* → 0 or 1
- ... and so on for all 6.

Since each person has **only one** income type, every row has **exactly one 1** and the rest are zeros. Hence the name *one-hot*: only one is "switched on".

This way no category ends up closer to or farther from another: they are all equally distinct from each other, and the three lies of the number line disappear.

### First, a single person

In [ ]:
from sklearn.preprocessing import OneHotEncoder

demo = OneHotEncoder(handle_unknown="ignore").fit(X_train[["NAME_INCOME_TYPE"]])
names = list(demo.get_feature_names_out())     # names of the new columns

# We grab any person and encode them.
person = X_train["NAME_INCOME_TYPE"].mode()[0]
row = demo.transform(pd.DataFrame({"NAME_INCOME_TYPE": [person]})).toarray()[0]

print(f"BEFORE ->  1 text column with the value '{person}'")
print(f"AFTER  -> {len(names)} columns of 0/1:\n")
print(pd.Series(row.astype(int), index=names).to_string())

The only column left at **1** is the one carrying the name of the value that person had. All the rest, at 0.

### Now, one person per type

To see the full pattern, we encode one person for each possible category. How to read the chart:

- **each row** = one person, and the label on the left is **what their original text column said**;
- **each column** = one of the 6 new columns;
- **each cell** = the answer for that person: blue with `1` = yes, grey with `0` = no.

In [ ]:
values = sorted(X_train["NAME_INCOME_TYPE"].unique())   # one person per category
matrix = demo.transform(pd.DataFrame({"NAME_INCOME_TYPE": values})).toarray()

short = [n.replace("NAME_INCOME_TYPE_", "") for n in names]
table = pd.DataFrame(matrix, columns=short, index=values)

fig, ax = plt.subplots(figsize=(1.5 * len(short) + 2, 0.55 * len(table) + 2.5))
sns.heatmap(
    table, annot=True, fmt=".0f", cbar=False, linewidths=1.5, linecolor="white",
    cmap=["#f0f0f0", BLUE], vmin=0, vmax=1,
    annot_kws={"color": "white", "weight": "bold"}, ax=ax,
)
ax.set_ylabel("what the original column said\n(one person per row)")
ax.set_xlabel("the NEW columns created by one-hot")
ax.set_title("1 text column  ->  6 columns of 0s and 1s", fontsize=12)
plt.xticks(rotation=30, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

Example of how to read it: the row labeled `Pensioner` has a **1** in the `Pensioner` column and **0** in the other five. That is all one-hot does.

> ⚠️ **The diagonal is an artifact of the example, not of the method.** We chose to show one person per category and in alphabetical order, so the ones end up aligned. With real data the rows come in any order and repeat (thousands of `Working` in a row), and no diagonal shows up.

Now for real, applied to **all** the multiclass columns at once:

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore")
ohe.fit(X_train[multiclass])           # <-- fit ONLY on train

train_ohe = ohe.transform(X_train[multiclass]).toarray()
val_ohe = ohe.transform(X_val[multiclass]).toarray()

print(f"{len(multiclass)} text columns  ->  {train_ohe.shape[1]} columns of 0/1")
print()
pd.DataFrame(train_ohe[:4], columns=ohe.get_feature_names_out(multiclass)).T.head(12)

### The important detail: `handle_unknown="ignore"`

Imagine that the occupation `Astronaut` never showed up in train, but it does appear in validation or in production. What does the encoder do?

- By default (`handle_unknown="error"`) → it **breaks** with a `ValueError`.
- With `handle_unknown="ignore"` → it puts **all zeros** in that block and moves on.

With real data this always happens, so the second option is the one we use.

In [ ]:
# We build a row with a category that does not exist in train.
new_row = X_train[multiclass].head(1).copy()
new_row.loc[:, multiclass[0]] = "CATEGORY_THAT_DOES_NOT_EXIST"

n_cat = len(ohe.categories_[0])   # how many columns that variable's block takes up
print(f"Modified column: {multiclass[0]}  ({n_cat} one-hot columns)")
print()
print("With handle_unknown='ignore', that block looks like this:")
print("  ", ohe.transform(new_row).toarray()[0][:n_cat], " -> all zeros, no failure")
print()
print("With the default behavior ('error'):")
try:
    OneHotEncoder().fit(X_train[multiclass]).transform(new_row)
except ValueError as e:
    print(f"   ValueError: {str(e)[:100]}...")

---

# Step 4 — Everything together

This is, literally, what step 2 of `preprocess_data()` does: ordinal for the binary ones, one-hot for the rest, and **always `fit` on train only**.

In [ ]:
def encode(train_df, val_df):
    """Ordinal for the binary ones, one-hot for the rest. Fit on train only."""
    cat = train_df.select_dtypes(include="object").columns
    card = train_df[cat].nunique()
    bina, multi = list(card[card == 2].index), list(card[card > 2].index)

    tr, va = train_df.copy(), val_df.copy()

    # 1) binary -> 0/1
    oe = OrdinalEncoder().fit(tr[bina])
    tr[bina], va[bina] = oe.transform(tr[bina]), oe.transform(va[bina])

    # 2) multiclass -> one column per category
    oh = OneHotEncoder(handle_unknown="ignore").fit(tr[multi])
    cols = oh.get_feature_names_out(multi)
    tr_oh = pd.DataFrame(oh.transform(tr[multi]).toarray(), columns=cols, index=tr.index)
    va_oh = pd.DataFrame(oh.transform(va[multi]).toarray(), columns=cols, index=va.index)

    # 3) we replace the text columns with the new ones
    tr = pd.concat([tr.drop(columns=multi), tr_oh], axis=1)
    va = pd.concat([va.drop(columns=multi), va_oh], axis=1)
    return tr, va


train_enc, val_enc = encode(X_train, X_val)

print(f"Before: {X_train.shape[1]} columns")
print(f"After:  {train_enc.shape[1]} columns")
print(f"Any text column left?  {(train_enc.dtypes == 'object').sum()}")
print(f"Do train and val have the same columns, in the same order?  "
      f"{list(train_enc.columns) == list(val_enc.columns)}")

### Where do all those new columns come from?

Each multiclass column turns into as many columns as it has categories. Added up, they explain the growth:

In [ ]:
# How many columns each one generates, according to what the encoder actually learned.
generated = pd.Series(
    {col: len(cats) for col, cats in zip(multiclass, ohe.categories_)}
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 0.45 * len(generated) + 2.2))

# Left: how many columns each categorical contributes.
axes[0].barh(generated.index, generated.values, color=BLUE)
for name, n in generated.items():
    axes[0].text(n + 0.1, name, str(n), va="center", fontsize=9)
axes[0].set_xlabel("new columns it generates")
axes[0].set_title("Each categorical expands into several columns")

# Right: total before vs after.
before_n, after_n = X_train.shape[1], train_enc.shape[1]
bars = axes[1].bar(["before", "after"], [before_n, after_n], color=["lightgrey", GREEN], width=0.5)
axes[1].bar_label(bars, fmt="%d", fontsize=13, weight="bold")
axes[1].set_ylabel("number of columns")
axes[1].set_title(f"Total features: {before_n} → {after_n}")

plt.tight_layout()
plt.show()

print("In the full project (with the 121 columns of the real dataset): 121 -> 246 features.")

> ⚠️ If a column has **missing values**, `OneHotEncoder` treats them as *just another category* and gives them their own column (that is why `OCCUPATION_TYPE`, which has nulls, generates an extra column). Missing values are handled in **notebook 05**.

### 🔑 The check that matters most

> *Do train and val have the same columns, in the same order?* → **True**

If you encoded train and validation **separately** (for example, `pd.get_dummies()` on each one), you could end up with **a different number of columns or a different order**. The model would then read values in the wrong column — and **it raises no error at all**. It simply works badly.

That is the reason for using sklearn's encoders: they **remember** the categories they saw in train and reproduce them identically in val and in test.

---

## Other techniques, worth mentioning in passing

| Technique | When it is used | Risk |
|---|---|---|
| **One-hot** | Few categories (< 15) | Column explosion |
| **Ordinal** | Variables with a **real** order (`Low < Medium < High`) | Inventing an order that does not exist |
| **Target encoding** | High cardinality (zip codes) | **Severe leakage** if not done with CV |
| **Frequency encoding** | High cardinality, simple alternative | Two equally frequent categories get confused |
| **Embeddings** | Very high cardinality, deep learning | Complexity |

`NAME_EDUCATION_TYPE` is the interesting case to discuss: it **does** have a natural order (`Lower secondary < Secondary < Incomplete higher < Higher education`), so a **properly configured** ordinal encoding would give the model more information than one-hot. The project uses one-hot for simplicity.

---

## Summary in 3 sentences

1. Models only read numbers, so text has to be translated.
2. With **2 categories**, `0/1` is enough; with **more than 2**, one column per category — because numbering them would invent an order and distances that do not exist.
3. The `fit` is done **on train only**, so that train, val and test end up with exactly the same columns.

---

## Exercises

1. Encode `NAME_EDUCATION_TYPE` with `OrdinalEncoder(categories=[[...]])` respecting the real educational order. Then compare (notebook 07) the AUC of a logistic regression with that encoding vs. one-hot.
2. What happens to the `NaN`s in `OCCUPATION_TYPE` when they go through the `OneHotEncoder`? Look at the resulting matrix: does a column show up for the nulls? What does that imply for notebook 05 (imputation)?
3. Encode `X_train` and `X_val` separately with `pd.get_dummies()` and compare `list(columns)` of each. Do they match? What would happen if you fed that to a model?

In [ ]:
# Your turn